<a href="https://colab.research.google.com/github/JoelRomero123/PARCIAL4-JOEL-ROMERO-2523552017/blob/main/clave_H_asociacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [5]:
url="https://raw.githubusercontent.com/JoelRomero123/PARCIAL4-JOEL-ROMERO-2523552017/refs/heads/main/clave_H_asociacion.csv"
df=pd.read_csv(url)
 #Mostrar las primeras filas del dataset y explicar su estructura.
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 624 entries, 0 to 623
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  624 non-null    object
 1   cliente_id      624 non-null    object
 2   fecha           624 non-null    object
 3   categoria       624 non-null    object
 4   item            624 non-null    object
 5   cantidad        624 non-null    int64 
 6   canal           623 non-null    object
dtypes: int64(1), object(6)
memory usage: 34.3+ KB


### Verificar valores nulos

In [8]:
print('Valores nulos por columna:')
print(df.isnull().sum())

Valores nulos por columna:
transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64


### Verificar valores duplicados

In [9]:
print('\nNúmero de filas duplicadas:')
print(df.duplicated().sum())


Número de filas duplicadas:
1


In [11]:
print('Tipos de datos actuales del DataFrame:')
print(df.dtypes)

Tipos de datos actuales del DataFrame:
transaccion_id    object
cliente_id        object
fecha             object
categoria         object
item              object
cantidad           int64
canal             object
dtype: object


### Preparación de Datos para Reglas de Asociación

Para aplicar algoritmos de reglas de asociación como Apriori, necesitamos transformar el dataset en un formato de lista de transacciones, donde cada transacción es una lista de ítems.

In [12]:
# Agrupar por 'transaccion_id' y recolectar todos los 'item' en una lista para cada transacción
transactions = df.groupby('transaccion_id')['item'].apply(list).reset_index()

# Mostrar las primeras transacciones para verificar el formato
display(transactions.head())

,transaccion_id,item
0,H-T0001,"[Backup_cloud, CRM, Certificacion, Notas]"
1,H-T0002,"[CRM, Certificacion, Curso_ingles]"
2,H-T0003,"[Almacenamiento, Backup_cloud, Dominio, Hosting]"
3,H-T0004,"[Almacenamiento, Gestor_password, Plan_deporte..."
4,H-T0005,"[Almacenamiento, Gestor_password, VPN, Almacen..."


### One-Hot Encoding para Algoritmo Apriori

Convertiremos la lista de ítems de cada transacción en un formato codificado (one-hot encoding) que es requerido por el algoritmo Apriori. Esto creará una matriz donde las filas son transacciones y las columnas son ítems, con valores booleanos indicando la presencia del ítem en la transacción.

In [13]:
from mlxtend.preprocessing import TransactionEncoder

# Obtener solo las listas de ítems de las transacciones
list_of_items = transactions['item'].tolist()

te = TransactionEncoder()
te_ary = te.fit(list_of_items).transform(list_of_items)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

display(df_encoded.head())

,Agenda,Almacenamiento,Antivirus,Backup_cloud,CRM,Certificacion,Colaboracion,Curso_excel,Curso_ingles,Curso_python,Dominio,Gestor_password,Hosting,Monitoreo,Notas,Plan_deportes,Plan_familiar,Plan_musica,Plan_video,VPN
0,False,False,False,True,True,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False
1,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False
2,False,True,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False
3,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False
4,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True


### Aplicar Algoritmo Apriori

Una vez que los datos están en el formato de one-hot encoding, podemos aplicar el algoritmo Apriori para encontrar los conjuntos de ítems frecuentes. Necesitamos definir un umbral mínimo de soporte (min_support) para filtrar los conjuntos de ítems que aparecen con una frecuencia mínima en el dataset.

In [14]:
from mlxtend.frequent_patterns import apriori

# Aplicar el algoritmo Apriori
frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)

# Mostrar los conjuntos de ítems frecuentes
display(frequent_itemsets.head())

,support,itemsets
0,0.095,(Agenda)
1,0.175,(Almacenamiento)
2,0.080,(Antivirus)
3,0.115,(Backup_cloud)
4,0.115,(CRM)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
from mlxtend.frequent_patterns import association_rules

# Generar reglas de asociación
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Mostrar las 10 reglas de asociación más relevantes, ordenadas por lift descendente
display(rules.sort_values('lift', ascending=False).head(10))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
15,(Gestor_password),(VPN),0.195,0.245,0.125,0.641026,2.616431,1.0,0.077225,2.103214,0.767453,0.396825,0.524537,0.575615
14,(VPN),(Gestor_password),0.245,0.195,0.125,0.510204,2.616431,1.0,0.077225,1.643542,0.818278,0.396825,0.391558,0.575615
12,(Dominio),(Hosting),0.185,0.225,0.105,0.567568,2.522523,1.0,0.063375,1.792188,0.740578,0.344262,0.442023,0.517117
13,(Hosting),(Dominio),0.225,0.185,0.105,0.466667,2.522523,1.0,0.063375,1.528125,0.778802,0.344262,0.345603,0.517117
17,(Plan_familiar),(Plan_video),0.215,0.265,0.140,0.651163,2.457218,1.0,0.083025,2.107000,0.755460,0.411765,0.525392,0.589732
16,(Plan_video),(Plan_familiar),0.265,0.215,0.140,0.528302,2.457218,1.0,0.083025,1.664200,0.806851,0.411765,0.399111,0.589732
7,(Curso_python),(Certificacion),0.240,0.255,0.140,0.583333,2.287582,1.0,0.078800,1.788000,0.740602,0.394366,0.440716,0.566176
6,(Certificacion),(Curso_python),0.255,0.240,0.140,0.549020,2.287582,1.0,0.078800,1.685217,0.755513,0.394366,0.406605,0.566176
4,(Certificacion),(CRM),0.255,0.115,0.050,0.196078,1.705030,1.0,0.020675,1.100854,0.555034,0.156250,0.091614,0.315431
5,(CRM),(Certificacion),0.115,0.255,0.050,0.434783,1.705030,1.0,0.020675,1.318077,0.467232,0.156250,0.241319,0.315431


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag